In [ ]:
import pandas as pd

def convert_image_path_names(data, mapping):
    # select the correct columns
    selected_data = data[['image_path', 'model_name']].copy()
    # map model names to integers
    selected_data['model_name'] = selected_data['model_name'].map(mapping)
    
    # replace image paths with correct numbers
    def replace_image_path(img_path):
        base = img_path.split('.')[0]  # '000-1'
        parts = base.split('-')       # ['000', '1']
        
        # p0 is the prompt number (0-99)
        p0 = int(parts[0].lstrip('0') or '0') 
        # p1 is the version number (1-4)
        p1 = int(parts[1])
        
        
        img_number = (p0 * 4) + (p1 - 1)
        
        # Check if the number is in the 0-399 range for each model
        if img_number < 0 or img_number > 399:
            # Corrected the error message range from 0-99 to 0-399
            raise ValueError(f"Image number {img_number} out of expected range (0-399)")
        
        return img_number 
    
    selected_data['image_path'] = selected_data['image_path'].apply(replace_image_path)
    
    # This part of your logic was correct:
    # Add the model offset (0, 400, 800, etc.)
    selected_data['image_path'] = selected_data['image_path'].astype(int) + (400 * selected_data['model_name'])
    
    # Format as .png
    selected_data['image_path'] = selected_data['image_path'].apply(lambda x: f"{x}.png")
    return selected_data['image_path']
    
    

if __name__ == "__main__":
    mapping = {
        'Controlnet': 0,
        'DALLE': 1,
        'Glide': 2,
        'Lafite': 3,
        'stable-diffusion': 4,  
        'Unidiffuser': 5
    }
    FILE_PATH = '01_scores.csv'
    # Load data from a CSV file
    data = pd.read_csv(FILE_PATH)
    column_names = ['id', 'presentation_order', 'image_path', 'model_name', 'quality_score', 'auth_score', 'text_corr_score']
    data.columns = column_names
    
    new_images_ids = convert_image_path_names(data, mapping)
    # add new image ids to the original dataframe
    data['new_image_id'] = new_images_ids
    # place it after image_path
    cols = data.columns.tolist()
    cols.insert(3, cols.pop(cols.index('new_image_id')))
    data = data[cols]
    
    # Save the updated dataframe to a new CSV file
    data.to_csv('01_scores_updated.csv', index=False)